# Visualization of Cells by Flow-Response PCA Score

## Objective
This notebook visualizes individual cells/objects based on their **flow-response PCA score** generated via sisPCA (Supervised Integrative Structural Principal Component Analysis).

## Method Background: Flow-Response Score Derivation via Supervised PCA (sisPCA)
We adapted the Supervised Integrative Structural Principal Component Analysis (sisPCA) method ([arXiv:2410.23595](https://doi.org/10.48550/arXiv.2410.23595)) to aggregate morphological features involved in EC response to shear stress. 

In brief:
- A dimension reduction model was fitted to extract a **single flow-response score per cell**
- The model was designed to **maximize separation between treatment groups** while **minimizing separation across biological and technical replicates**
- Flow exposure was encoded as a continuous variable: 0 (static), 1 (2 hr flow), and 5 (24 hr flow)
- A batch effect penalty (l=5) was included in the model
- The supervised PCA model was **trained only on WT cells**
- The trained model was subsequently used to compute flow-response scores for **all cells, including CCM2-V53I mutant cells**

## Experimental Setup
- **Genotypes**: CCM2 WT and coding variant V53I
- **Conditions**: STATIC / 2HR of shear stress / 24HR of shear stress
- **Biological replicates**: 
  - WT: W3, W7, W11
  - V53I: V4, V8, V12
- **Imaging**: Each biological replicate seeded in one chamber slide with ~10 images of different views
- **Input files**: CZI format (Zeiss)
- **Channels**:
  - DAPI = nucleus
  - NIR = WGA-770
  - AF488 = phalloidin
  - AF647 = GM130 (Golgi)
  - AF568 = CCM2-V5

## Analysis Goals
1. **WT-only visualization**: Show that PCA score depicts morphological flow response across 3 conditions
2. **WT vs V53I comparison**: Compare WT and V53I under 24HR shear stress
3. **Statistical analysis**: Density plots with statistical comparisons

## 1. Setup and Imports

In [ ]:
# Core data manipulation
import polars as pl
import pandas as pd
import numpy as np

# Image processing
from skimage.io import imread
from skimage.transform import resize
from scipy.ndimage import gaussian_filter

# Visualization
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns

# File system
from pathlib import Path
import glob
import os

# Specialized CZI support
from aicspylibczi import CziFile

# Statistics
from scipy import stats
from statsmodels.stats.multitest import multipletests

# Warnings
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.size'] = 10

## 2. Data Paths and Configuration

In [ ]:
# Flow PCA scores CSV
# Update this path to point to your sisPCA output file
FLOW_PCA_CSV = "data/flow_pc_scores.csv"

# Output directory
OUTPUT_DIR = "results/flowscore_visualization"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Image directories (CZI files)
# Update these paths to point to your local CZI image directories
IMG_DIRS = [
    "data/images/batch1/",
    "data/images/batch2/",
    "data/images/batch3/"
]

# Crop parameters
CROP_SIZE = 1024  # 1024 pixels total (512 from center in each direction)

# Random seed for reproducibility
RANDOM_SEED = 42

## 3. Load Flow PCA Scores Data

In [ ]:
# Load data with Polars for efficiency
df = pl.read_csv(FLOW_PCA_CSV)

print(f"Total cells loaded: {len(df):,}")
print(f"\nColumns: {df.columns[:10]}... ({len(df.columns)} total)")
print(f"\nKey metadata columns:")
print(f"  - Genotypes: {df['genotype'].unique().to_list()}")
print(f"  - Treatments: {df['treatment'].unique().to_list()}")
print(f"  - Biological replicates: {df['biorep'].unique().to_list()}")
print(f"\nFlow PC score range: [{df['flow_pc_score_all'].min():.2f}, {df['flow_pc_score_all'].max():.2f}]")

# Display sample
df.head()

In [ ]:
# Check cell counts per condition and biorep
print("\n" + "="*80)
print("CELL COUNT ANALYSIS: Cells per Condition × Biorep")
print("="*80)

for genotype in ["WT", "V53I"]:
    print(f"\n{genotype}:")
    print("-" * 60)
    for condition in ["STATIC", "2HR", "24HR"]:
        print(f"  {condition}:")
        for biorep in ["biorep-1", "biorep-2", "biorep-3"]:
            count = len(df.filter(
                (pl.col("genotype") == genotype) &
                (pl.col("treatment") == condition) &
                (pl.col("biorep") == biorep)
            ))
            print(f"    {biorep}: {count} cells")


## 4. Helper Functions for Image Processing

### 4.1 CZI File Handling

In [ ]:
def find_image_file(filename, img_dirs=IMG_DIRS):
    """
    Find CZI file in the image directories.
    
    Parameters:
    -----------
    filename : str
        Base filename (e.g., 'SLIDE10_V4_2HR_01_B2Im1.czi')
    img_dirs : list
        List of directories to search
        
    Returns:
    --------
    Path or None
    """
    for img_dir in img_dirs:
        matches = list(Path(img_dir).rglob(filename))
        if matches:
            return matches[0]
    return None


def channel_to_cmap(channel):
    """Map channel name to colormap."""
    if channel == "CCM2":
        cmap = mpl.colors.LinearSegmentedColormap.from_list("orange_cmap", ["#000","#FF8C00"])  # Orange for AF568
    elif channel == "DAPI":
        cmap = mpl.colors.LinearSegmentedColormap.from_list("blue_cmap", ["#000","#0000FF"])
    elif channel == "GM130":
        cmap = mpl.colors.LinearSegmentedColormap.from_list("red_cmap", ["#000","#FF0000"]) 
    elif channel == "Phalloidin":
        cmap = mpl.colors.LinearSegmentedColormap.from_list("magenta_cmap", ["#000","#FF00FF"])
    elif channel == "WGA":
        cmap = mpl.colors.LinearSegmentedColormap.from_list("cyan_cmap", ["#000","#00FFFF"])
    else:
        cmap = "gray"
    return cmap


def read_czi_channel(filepath, channel_name="CCM2"):
    """
    Read a specific channel from CZI file.
    
    Parameters:
    -----------
    filepath : Path or str
        Path to CZI file
    channel_name : str
        Channel name: 'DAPI', 'WGA', 'Phalloidin', 'GM130', or 'CCM2'
        
    Returns:
    --------
    np.ndarray : 2D image array
    """
    # Channel index mapping, always double check with microscope setup and the cellprofiler pipeline
    channel_indices = {
        "DAPI": 0,
        "CCM2": 4,  
        "GM130": 3,
        "Phalloidin": 2,
        "WGA": 1
    }
    
    try:
        czi = CziFile(str(filepath))
        img_array = np.squeeze(czi.read_image()[0])
        
        ch_idx = channel_indices[channel_name]
        # Extract 2D slice from multi-dimensional array
        img_2d = img_array[ch_idx, :, :]  # Takes first Z-slice
        
        return img_2d.astype(np.float32)
    except Exception as e:
        print(f"Error reading {filepath}, channel {channel_name}: {e}")
        return None


def read_all_channels(filepath):
    """
    Read all 5 channels from CZI file.
    
    Returns:
    --------
    dict : {channel_name: 2D array}
    """
    channels = {}
    for ch_name in ["DAPI", "WGA", "Phalloidin", "GM130", "CCM2"]:
        channels[ch_name] = read_czi_channel(filepath, ch_name)
    return channels

### 4.2 Image Cropping and Enhancement

In [ ]:
def crop_cell_from_image(img, center_x, center_y, crop_size=1024):
    """
    Crop 1024x1024 pixels centered on cell coordinates.
    
    Parameters:
    -----------
    img : np.ndarray
        Full image array
    center_x, center_y : int
        Cell center coordinates
    crop_size : int
        Total crop size (default 1024)
        
    Returns:
    --------
    np.ndarray : Cropped image
    """
    half_crop = crop_size // 2  # 512 pixels
    
    # Calculate crop boundaries
    x_min = max(0, int(center_x - half_crop))
    x_max = min(img.shape[1], int(center_x + half_crop))
    y_min = max(0, int(center_y - half_crop))
    y_max = min(img.shape[0], int(center_y + half_crop))
    
    # Crop
    img_crop = img[y_min:y_max, x_min:x_max]
    
    # Resize to exact size if at image edges
    if img_crop.shape != (crop_size, crop_size):
        img_crop = resize(img_crop, (crop_size, crop_size), 
                         preserve_range=True, anti_aliasing=True)
    
    return img_crop.astype(np.float32)


def enhance_image(img, bg_percentile=10, blur_sigma=1.0, vmax_percentile=99.9):
    """
    Apply image enhancement pipeline:
    1. Background subtraction
    2. Gaussian smoothing
    3. Contrast stretching
    
    Parameters:
    -----------
    img : np.ndarray
        Input image
    bg_percentile : float
        Percentile for background subtraction (default 10)
    blur_sigma : float
        Gaussian blur sigma (default 1.0)
    vmax_percentile : float
        Percentile for contrast stretching (default 99.9)
        
    Returns:
    --------
    tuple : (enhanced_img, vmax)
    """
    # Background subtraction
    bg = np.percentile(img, bg_percentile)
    img_enhanced = np.clip(img - bg, 0, None)
    
    # Gaussian smoothing
    img_enhanced = gaussian_filter(img_enhanced, sigma=blur_sigma)
    
    # Contrast stretching
    vmax = np.percentile(img_enhanced, vmax_percentile)
    
    return img_enhanced, vmax

### 4.3 Channel Merging and Color Mapping

In [ ]:
def create_rgb_composite(channels_dict, crop_x, crop_y, crop_size=1024):
    """
    Create RGB composite image from all 5 channels.
    
    Color mapping:
    - DAPI (nucleus): Blue
    - WGA-770: Magenta (tuned down)
    - Phalloidin: Green (tuned down)
    - GM130 (Golgi): Red (enhanced for shear stress response)
    - CCM2-V5: Orange
    
    Parameters:
    -----------
    channels_dict : dict
        Dictionary of channel arrays {channel_name: array}
    crop_x, crop_y : int
        Cell center coordinates
    crop_size : int
        Crop size in pixels
        
    Returns:
    --------
    np.ndarray : RGB image (H, W, 3)
    """
    # Initialize RGB channels
    rgb = np.zeros((crop_size, crop_size, 3), dtype=np.float32)
    
    # Process each channel
    for ch_name, ch_img in channels_dict.items():
        if ch_img is None:
            continue
            
        # Crop cell
        ch_crop = crop_cell_from_image(ch_img, crop_x, crop_y, crop_size)
        
        # Enhance
        ch_enhanced, vmax = enhance_image(ch_crop)
        
        # Normalize to [0, 1]
        if vmax > 0:
            ch_norm = np.clip(ch_enhanced / vmax, 0, 1)
        else:
            ch_norm = ch_enhanced
        
        # Add to RGB with appropriate color
        if ch_name == "DAPI":
            # Blue
            rgb[:, :, 2] += ch_norm * 0.8
        elif ch_name == "WGA":
            # Magenta (red + blue) - tuned down
            rgb[:, :, 0] += ch_norm * 0.5
            rgb[:, :, 2] += ch_norm * 0.5
        elif ch_name == "Phalloidin":
            # Green (tuned down)
            rgb[:, :, 1] += ch_norm * 0.5
        elif ch_name == "GM130":
            # Red (enhanced for Golgi positioning/shear stress response)
            rgb[:, :, 0] += ch_norm * 1.2
        elif ch_name == "CCM2":
            # Orange (red + green)
            rgb[:, :, 0] += ch_norm * 1.0
            rgb[:, :, 1] += ch_norm * 0.55
    
    # Clip to valid range
    rgb = np.clip(rgb, 0, 1)
    
    return rgb


def get_channel_colormap(channel_name):
    """
    Get matplotlib colormap for single-channel display.
    """
    if channel_name == "CCM2":
        return mpl.colors.LinearSegmentedColormap.from_list("orange_cmap", ["#000","#FF8C00"])  # Orange for AF568
    elif channel_name == "DAPI":
        return mpl.colors.LinearSegmentedColormap.from_list("blue_cmap", ["#000","#0000FF"])
    elif channel_name == "GM130":
        return mpl.colors.LinearSegmentedColormap.from_list("red_cmap", ["#000","#FF0000"]) 
    elif channel_name == "Phalloidin":
        return mpl.colors.LinearSegmentedColormap.from_list("green_cmap", ["#000","#00FF00"])
    elif channel_name == "WGA":
        return mpl.colors.LinearSegmentedColormap.from_list("magenta_cmap", ["#000","#FF00FF"])
    else:
        return "gray"

### 4.4 Cell Selection by Flow Score Percentiles

In [ ]:
def select_cells_by_percentile(df, genotype, treatment, bioreps, 
                               percentile_low, percentile_high, 
                               n_samples=3, seed=RANDOM_SEED):
    """
    Select cells within a percentile range of flow PC scores.
    
    Parameters:
    -----------
    df : polars.DataFrame
        Full dataset
    genotype : str
        'WT' or 'V53I'
    treatment : str
        'STATIC', '2HR', or '24HR'
    bioreps : list
        List of biological replicates (e.g., ['W3', 'W7', 'W11'])
    percentile_low, percentile_high : float
        Percentile range (0-100)
    n_samples : int
        Number of cells to sample from the range
    seed : int
        Random seed
        
    Returns:
    --------
    polars.DataFrame : Selected cells
    """
    # Filter by genotype, treatment, and bioreps
    filtered = df.filter(
        (pl.col("genotype") == genotype) &
        (pl.col("treatment") == treatment) &
        (pl.col("biorep").is_in(bioreps))
    )
    
    # Calculate percentile thresholds
    score_low = filtered["flow_pc_score_all"].quantile(percentile_low / 100)
    score_high = filtered["flow_pc_score_all"].quantile(percentile_high / 100)
    
    # Select cells in range
    pool = filtered.filter(
        (pl.col("flow_pc_score_all") >= score_low) &
        (pl.col("flow_pc_score_all") < score_high)
    )
    
    # Sample
    if len(pool) >= n_samples:
        selected = pool.sample(n=n_samples, seed=seed)
    else:
        selected = pool
        print(f"Warning: Only {len(pool)} cells available (requested {n_samples})")
    
    return selected

## 5. Visualization Set 1: WT-Only Flow Response

### Objective
Showcase that PCA score can depict morphological flow response in WT cells across three conditions.

### Design
- **Genotypes**: WT only (all biological replicates)
- **Conditions**: STATIC, 2HR, 24HR (3 separate visualizations)
- **Percentile Calculation**: Based on **ALL WT cells from all three conditions combined**
- **Percentile Ranges**: 
  - Low: 5th to 15th percentile
  - Middle: 45th to 55th percentile
  - High: 85th to 95th percentile
- **Cell Selection Strategy**: 
  - Target 12 cells per condition within each percentile range
  - If fewer than 12 cells available for a condition, select all available cells
- **Layout**: Each condition displayed separately in 2 columns × variable rows
  - 2 columns (6 cells per column)
  - Rows = 6 rows per percentile range × 3 percentile ranges = 18 rows total
  - Output: 3 separate high-resolution images (one per condition)

### Key Features
- **Global percentiles**: By calculating percentiles from ALL WT cells across all conditions, we ensure that the score ranges are consistent and biologically meaningful, allowing direct comparison of cell morphology across different shear stress exposures
- **Compact layout**: Two-column layout prevents excessively long images while maintaining high resolution (300 DPI)

In [ ]:
# Configuration for WT-only visualization
WT_BIOREPS = ["biorep-1", "biorep-2", "biorep-3"]
CONDITIONS = ["STATIC", "2HR", "24HR"]

# Updated percentile ranges: 5-15th, 45-55th, 85-95th
PERCENTILE_RANGES = [
    (5, 15, "5-15th"),
    (45, 55, "45-55th"),
    (85, 95, "85-95th")
]

N_CELLS_PER_CONDITION = 12  # Target 12 cells per condition in each percentile range

# STEP 1: Get ALL WT cells from all three conditions to calculate global percentiles
print("STEP 1: Calculating percentiles based on ALL WT cells from STATIC, 2HR, and 24HR")
print("=" * 80)

all_wt_cells = df.filter(
    (pl.col("genotype") == "WT") &
    (pl.col("treatment").is_in(["STATIC", "2HR", "24HR"]))
)

print(f"Total WT cells across all conditions: {len(all_wt_cells):,}")
print(f"Flow PC score range: [{all_wt_cells['flow_pc_score_all'].min():.2f}, {all_wt_cells['flow_pc_score_all'].max():.2f}]")

# Calculate global percentile thresholds from ALL WT cells
percentile_thresholds = {}
for pct_low, pct_high, label in PERCENTILE_RANGES:
    score_low = all_wt_cells["flow_pc_score_all"].quantile(pct_low / 100)
    score_high = all_wt_cells["flow_pc_score_all"].quantile(pct_high / 100)
    percentile_thresholds[label] = (score_low, score_high)
    print(f"\n{label} percentile: score range [{score_low:.2f}, {score_high:.2f}]")

# STEP 2: Select cells for each condition and percentile range
print("\n\nSTEP 2: Selecting cells for each condition and percentile range")
print("=" * 80)

# Structure: wt_selections[percentile_label][condition] = list of cells
wt_selections = {}

for pct_low, pct_high, label in PERCENTILE_RANGES:
    wt_selections[label] = {}
    score_low, score_high = percentile_thresholds[label]
    
    print(f"\n{label} Percentile Range [{score_low:.2f}, {score_high:.2f}]:")
    print("-" * 60)
    
    for condition in CONDITIONS:
        # Filter cells by condition and score range
        cells_in_range = df.filter(
            (pl.col("genotype") == "WT") &
            (pl.col("treatment") == condition) &
            (pl.col("flow_pc_score_all") >= score_low) &
            (pl.col("flow_pc_score_all") < score_high)
        )
        
        n_available = len(cells_in_range)
        
        # Select cells: 12 if available, otherwise all cells in this range
        if n_available >= N_CELLS_PER_CONDITION:
            selected = cells_in_range.sample(n=N_CELLS_PER_CONDITION, seed=RANDOM_SEED)
            print(f"  {condition}: Selected {N_CELLS_PER_CONDITION} cells (from {n_available} available)")
        else:
            selected = cells_in_range
            print(f"  {condition}: Selected ALL {n_available} cells (< {N_CELLS_PER_CONDITION} requested)")
        
        wt_selections[label][condition] = selected

print("\n" + "=" * 80)
print("Selection complete!")
print("=" * 80)

In [ ]:
def visualize_wt_flow_response_by_condition(selections_dict, percentile_ranges, crop_size=1024, save_path=None):
    """
    Create visualization grid for WT flow response across all three conditions.
    Shows three percentile ranges (5-15th, 45-55th, 85-95th) for all three conditions.
    
    Layout: Each condition shown in 2 columns to prevent excessively long images
    Variable rows based on cell count (up to 6 cells per column × 3 percentile ranges)
    
    Parameters:
    -----------
    selections_dict : dict
        Nested dict with structure: selections[percentile_label][condition]
    percentile_ranges : list
        List of tuples: (pct_low, pct_high, label)
    crop_size : int
        Size of cell crops in pixels
    save_path : str or None
        Path to save the figure
    """
    conditions = ["STATIC", "2HR", "24HR"]
    
    # Create separate figure for each condition
    for condition in conditions:
        print(f"\nGenerating visualization for {condition}...")
        
        # Collect all cells for this condition across percentile ranges
        # Organize as: percentile_ranges × 2 columns (6 cells per column)
        
        # Determine max cells per percentile for this condition
        max_cells_in_condition = 0
        for _, _, label in percentile_ranges:
            n_cells = len(selections_dict[label][condition])
            max_cells_in_condition = max(max_cells_in_condition, n_cells)
        
        # Calculate layout: 2 columns, rows = ceil(max_cells / 2) per percentile
        cells_per_col = 6
        n_cols = 2
        n_rows_per_percentile = cells_per_col
        n_rows = n_rows_per_percentile * len(percentile_ranges)
        
        # Create figure
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(10, n_rows * 2.5))
        
        # Ensure axes is 2D
        if n_rows == 1:
            axes = axes.reshape(1, -1)
        
        # Iterate through each percentile range
        for pct_idx, (pct_low, pct_high, label) in enumerate(percentile_ranges):
            cells = selections_dict[label][condition]
            cells_list = list(cells.iter_rows(named=True))
            
            # Display up to 12 cells (6 per column)
            for cell_idx in range(cells_per_col * n_cols):
                # Calculate position in grid
                row_in_percentile = cell_idx % cells_per_col
                col = cell_idx // cells_per_col
                
                row_idx = pct_idx * n_rows_per_percentile + row_in_percentile
                col_idx = col
                
                ax = axes[row_idx, col_idx]
                
                # Add percentile range label on the first cell of each range (left column only)
                if row_in_percentile == 0 and col_idx == 0:
                    ax.text(-0.2, 0.5, f"{label}\nPercentile", 
                           transform=ax.transAxes,
                           rotation=90,
                           verticalalignment='center',
                           horizontalalignment='right',
                           fontsize=11,
                           fontweight='bold')
                
                if cell_idx < len(cells_list):
                    cell = cells_list[cell_idx]
                    
                    try:
                        # Find image file
                        img_filename = cell["FileName_CCM2"]
                        img_path = find_image_file(img_filename)
                        
                        if img_path is None:
                            ax.text(0.5, 0.5, "Image\nnot found", 
                                   ha='center', va='center', fontsize=8)
                            ax.axis('off')
                            continue
                        
                        # Read all channels
                        channels = read_all_channels(img_path)
                        
                        # Create RGB composite
                        center_x = cell["Location_Center_X"]
                        center_y = cell["Location_Center_Y"]
                        rgb = create_rgb_composite(channels, center_x, center_y, crop_size)
                        
                        # Display - NO LABELS ON IMAGE (per user request)
                        ax.imshow(rgb)
                        
                    except Exception as e:
                        print(f"Error processing cell: {e}")
                        ax.text(0.5, 0.5, "Error", ha='center', va='center', fontsize=8)
                else:
                    # Empty cell - no data available
                    pass
                
                ax.axis('off')
        
        # Overall title for this condition
        plt.suptitle(f'WT Cells - {condition} Condition\n' +
                    'Flow-Response Score Percentiles (Based on ALL WT Cells)\n' +
                    'All 5 Channels Merged (DAPI=Blue, WGA=Magenta, Phalloidin=Green, GM130=Red, CCM2=Orange)',
                    fontsize=13, fontweight='bold', y=0.998)
        
        plt.tight_layout(rect=[0, 0, 1, 0.995])
        
        if save_path:
            # Create condition-specific filename
            condition_save_path = save_path.replace('.png', f'_{condition}.png')
            plt.savefig(condition_save_path, dpi=300, bbox_inches='tight')
            print(f"Saved to {condition_save_path}")
        
        plt.show()


# Generate visualization for each condition separately
print("\nGenerating WT flow response visualizations...")
visualize_wt_flow_response_by_condition(
    wt_selections,
    percentile_ranges=PERCENTILE_RANGES,
    crop_size=CROP_SIZE,
    save_path=f"{OUTPUT_DIR}/WT_flow_response.png"
)

In [ ]:
def visualize_wt_by_percentile_across_conditions(selections_dict, percentile_ranges, crop_size=1024, save_path=None):
    """
    Create visualization showing WT cells from ALL THREE conditions together, grouped by percentile range.
    
    Layout: Each percentile range gets 3 rows to show 12 cells per condition
    - 3 percentile ranges × 3 rows each = 9 rows total
    - Each row shows cells from all 3 conditions (STATIC, 2HR, 24HR)
    - 12 cells per condition displayed across 3 rows (4 cells per row)
    
    Parameters:
    -----------
    selections_dict : dict
        Nested dict with structure: selections[percentile_label][condition]
    percentile_ranges : list
        List of tuples: (pct_low, pct_high, label)
    crop_size : int
        Size of cell crops in pixels
    save_path : str or None
        Path to save the figure
    """
    conditions = ["STATIC", "2HR", "24HR"]
    n_cells_per_condition = 12  # 12 cells per condition
    n_cols_per_condition = 4    # 4 cells per condition per row
    n_cols = n_cols_per_condition * len(conditions)  # 12 columns total
    n_rows_per_percentile = 3   # 3 rows per percentile range (12 cells / 4 per row = 3 rows)
    n_rows = len(percentile_ranges) * n_rows_per_percentile  # 9 rows total
    
    # Create figure
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(24, n_rows * 2.5))
    
    # Ensure axes is 2D
    if n_rows == 1:
        axes = axes.reshape(1, -1)
    
    # Iterate through each percentile range
    for pct_idx, (pct_low, pct_high, label) in enumerate(percentile_ranges):
        # Each percentile range occupies 3 rows
        start_row = pct_idx * n_rows_per_percentile
        
        # Add percentile range label on the left (spanning all 3 rows)
        axes[start_row + 1, 0].text(-0.3, 0.5, f"{label}\nPercentile", 
                                   transform=axes[start_row + 1, 0].transAxes,
                                   rotation=90,
                                   verticalalignment='center',
                                   horizontalalignment='right',
                                   fontsize=12,
                                   fontweight='bold')
        
        # Iterate through each condition
        for cond_idx, condition in enumerate(conditions):
            cells = selections_dict[label][condition]
            cells_list = list(cells.iter_rows(named=True))
            
            # Add condition label at the top (first row of first percentile range only)
            if pct_idx == 0:
                col_center = cond_idx * n_cols_per_condition + n_cols_per_condition // 2
                axes[0, col_center].set_title(condition, fontsize=13, fontweight='bold', pad=10)
            
            # Display 12 cells for this condition across 3 rows (4 cells per row)
            for cell_idx in range(n_cells_per_condition):
                row_offset = cell_idx // n_cols_per_condition  # 0, 1, 2 -> which of the 3 rows
                col_offset = cell_idx % n_cols_per_condition   # 0-3 position within the row
                
                row_idx = start_row + row_offset
                col_idx = cond_idx * n_cols_per_condition + col_offset
                
                ax = axes[row_idx, col_idx]
                
                if cell_idx < len(cells_list):
                    cell = cells_list[cell_idx]
                    
                    try:
                        # Find image file
                        img_filename = cell["FileName_CCM2"]
                        img_path = find_image_file(img_filename)
                        
                        if img_path is None:
                            ax.text(0.5, 0.5, "Image\nnot found", 
                                   ha='center', va='center', fontsize=7)
                            ax.axis('off')
                            continue
                        
                        # Read all channels
                        channels = read_all_channels(img_path)
                        
                        # Create RGB composite
                        center_x = cell["Location_Center_X"]
                        center_y = cell["Location_Center_Y"]
                        rgb = create_rgb_composite(channels, center_x, center_y, crop_size)
                        
                        # Display - NO LABELS ON IMAGE
                        ax.imshow(rgb)
                        
                    except Exception as e:
                        print(f"Error processing cell: {e}")
                        ax.text(0.5, 0.5, "Error", ha='center', va='center', fontsize=7)
                else:
                    # Empty cell - no data available
                    pass
                
                ax.axis('off')
    
    # Overall title
    plt.suptitle('WT Cells: Flow-Response Across All Three Conditions\n' +
                'Grouped by Percentile Range (Based on ALL WT Cells)\n' +
                'All 5 Channels Merged (DAPI=Blue, WGA=Magenta, Phalloidin=Green, GM130=Red, CCM2=Orange)',
                fontsize=14, fontweight='bold', y=0.995)
    
    plt.tight_layout(rect=[0, 0, 1, 0.99])
    
    if save_path:
        save_path_combined = save_path.replace('.png', '_all_conditions.png')
        plt.savefig(save_path_combined, dpi=300, bbox_inches='tight')
        print(f"Saved to {save_path_combined}")
    
    plt.show()


# Generate visualization showing all conditions together
print("\nGenerating WT cells across all three conditions (grouped by percentile)...")
visualize_wt_by_percentile_across_conditions(
    wt_selections,
    percentile_ranges=PERCENTILE_RANGES,
    crop_size=CROP_SIZE,
    save_path=f"{OUTPUT_DIR}/WT_flow_response.png"
)

## 6. Visualization Set 2: WT vs V53I Under 24HR Shear Stress

### Objective
Compare WT and V53I cells under 24HR shear stress condition.

### Design
- **Genotypes**: WT and V53I
- **Condition**: 24HR only
- **Bioreps**: 
  - WT: W3, W7, W11
  - V53I: V4, V8, V12
- **Score ranges**: 
  - Low: 5th to 15th percentile
  - High: 80th to 95th percentile
- **Images**: 3 per genotype per range
- **Total**: 2 genotypes × 2 ranges × 3 images = 12 cell crops

### Layout
4 rows × 3 columns:
- Row 1: WT Low (5-15th)
- Row 2: WT High (80-95th)
- Row 3: V53I Low (5-15th)
- Row 4: V53I High (80-95th)

In [ ]:
# Configuration for WT vs V53I comparison
WT_BIOREPS = ["biorep-1", "biorep-2", "biorep-3"]
V53I_BIOREPS = ["biorep-1", "biorep-2", "biorep-3"]
COMPARISON_CONDITION = "24HR"
N_COMPARISON_IMAGES = 6  # 6 images: 2 per biorep (3 bioreps)

# Select cells
comparison_selections = {}

# WT Low (5-15th percentile)
comparison_selections["WT_low"] = select_cells_by_percentile(
    df, genotype="WT", treatment=COMPARISON_CONDITION, bioreps=WT_BIOREPS,
    percentile_low=5, percentile_high=15, n_samples=N_COMPARISON_IMAGES
)

# WT High (80-95th percentile)
comparison_selections["WT_high"] = select_cells_by_percentile(
    df, genotype="WT", treatment=COMPARISON_CONDITION, bioreps=WT_BIOREPS,
    percentile_low=80, percentile_high=95, n_samples=N_COMPARISON_IMAGES
)

# V53I Low (5-15th percentile)
comparison_selections["V53I_low"] = select_cells_by_percentile(
    df, genotype="V53I", treatment=COMPARISON_CONDITION, bioreps=V53I_BIOREPS,
    percentile_low=5, percentile_high=15, n_samples=N_COMPARISON_IMAGES
)

# V53I High (80-95th percentile)
comparison_selections["V53I_high"] = select_cells_by_percentile(
    df, genotype="V53I", treatment=COMPARISON_CONDITION, bioreps=V53I_BIOREPS,
    percentile_low=80, percentile_high=95, n_samples=N_COMPARISON_IMAGES
)

# Print selections
for key, cells in comparison_selections.items():
    print(f"{key}: {len(cells)} cells selected")

In [ ]:
def visualize_wt_vs_v53i_comparison(selections_dict, crop_size=1024, save_path=None):
    """
    Create comparison grid for WT vs V53I under 24HR shear stress.
    
    Layout: 8 rows (4 groups × 2 subrows) × 3 columns (3 bioreps)
    Each group has 6 images: 2 rows × 3 bioreps
    """
    groups = ["WT_low", "WT_high", "V53I_low", "V53I_high"]
    group_labels = [
        "WT - Low (5-15th)",
        "WT - High (80-95th)",
        "V53I - Low (5-15th)",
        "V53I - High (80-95th)"
    ]
    
    n_bioreps = 3
    
    # Create figure: 8 rows (4 groups × 2) × 3 columns (3 bioreps)
    fig, axes = plt.subplots(8, 3, figsize=(12, 32))
    
    for group_idx, (group, label) in enumerate(zip(groups, group_labels)):
        cells = selections_dict[group]
        cells_list = list(cells.iter_rows(named=True))
        
        # Sort by biorep to ensure consistent ordering
        cells_list.sort(key=lambda x: x["biorep"])
        
        for img_idx in range(6):  # 6 images total per group
            # Calculate position: 2 rows per group, 3 columns (bioreps)
            row_offset = group_idx * 2  # Each group takes 2 rows
            sub_row = img_idx // n_bioreps  # 0 or 1 (first or second row)
            biorep_col = img_idx % n_bioreps  # 0, 1, or 2 (which biorep)
            
            row_idx = row_offset + sub_row
            col_idx = biorep_col
            
            ax = axes[row_idx, col_idx]
            
            if img_idx < len(cells_list):
                cell = cells_list[img_idx]
                
                try:
                    # Find image file
                    img_filename = cell["FileName_CCM2"]
                    img_path = find_image_file(img_filename)
                    
                    if img_path is None:
                        ax.text(0.5, 0.5, "Image not found", 
                               ha='center', va='center', fontsize=10)
                        ax.axis('off')
                        continue
                    
                    # Read all channels
                    channels = read_all_channels(img_path)
                    
                    # Create RGB composite
                    center_x = cell["Location_Center_X"]
                    center_y = cell["Location_Center_Y"]
                    rgb = create_rgb_composite(channels, center_x, center_y, crop_size)
                    
                    # Display
                    ax.imshow(rgb)
                    
                    # Add label - only show genotype and flow score (no biorep)
                    flow_score = cell["flow_pc_score_all"]
                    genotype = cell["genotype"]
                    label_text = f"{genotype}\n{flow_score:.2f}"
                    
                    # Color code by genotype
                    label_color = '#4169E1' if genotype == "WT" else '#FF8C00'
                    
                    ax.text(0.05, 0.95, label_text, 
                           transform=ax.transAxes,
                           color='white', fontsize=8,
                           verticalalignment='top',
                           bbox=dict(facecolor=label_color, alpha=0.8, 
                                   linewidth=0, pad=3))
                    
                    # Add group label on left column (only on first subrow of each group)
                    if col_idx == 0 and sub_row == 0:
                        ax.set_ylabel(label, fontsize=11, fontweight='bold',
                                    rotation=0, ha='right', va='center', labelpad=15)
                    
                except Exception as e:
                    print(f"Error processing cell: {e}")
                    ax.text(0.5, 0.5, "Error", ha='center', va='center', fontsize=10)
            
            ax.axis('off')
    
    # Overall title
    plt.suptitle('WT vs V53I Comparison Under 24HR Shear Stress\n' +
                'All 5 Channels Merged (DAPI=Blue, WGA=Magenta, Phalloidin=Green, GM130=Red, CCM2=Orange)',
                fontsize=14, fontweight='bold', y=0.995)
    
    plt.tight_layout(rect=[0, 0, 1, 0.99])
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Saved to {save_path}")
    
    plt.show()

# Generate visualization
visualize_wt_vs_v53i_comparison(
    comparison_selections,
    crop_size=CROP_SIZE,
    save_path=f"{OUTPUT_DIR}/WT_vs_V53I_24HR_comparison.png"
)

## 6.2 Alternative Strategy: WT vs V53I with Global Percentiles

### Objective
Compare WT and V53I cells using percentiles calculated from **both genotypes combined** under 24HR shear stress.

### Design
- **Percentile Calculation**: Based on **ALL cells (both WT and V53I) under 24HR condition**
- **Percentile Ranges**: 
  - Low: 5th to 15th percentile
  - High: 85th to 95th percentile
- **Cell Selection**: 12 cells per genotype in each percentile range
- **Total**: 2 genotypes × 2 ranges × 12 cells = 48 cell images
- **Layout**: 8 rows × 6 columns
  - Rows 1-2: WT Low (12 cells)
  - Rows 3-4: V53I Low (12 cells)
  - Rows 5-6: WT High (12 cells)
  - Rows 7-8: V53I High (12 cells)
- **Resolution**: 300 DPI

### Rationale
This approach ensures that WT and V53I cells are compared on the same absolute scale, revealing whether the genotypes differ in their distribution across the global flow-response spectrum.

In [ ]:
# Calculate global percentiles from BOTH WT and V53I cells under 24HR condition
print("Calculating global percentiles from both genotypes under 24HR condition...")
print("=" * 80)

# Get all 24HR cells (both genotypes)
all_24hr_cells = df.filter(
    (pl.col("treatment") == "24HR")
)

print(f"Total 24HR cells (WT + V53I): {len(all_24hr_cells):,}")
print(f"Flow PC score range: [{all_24hr_cells['flow_pc_score_all'].min():.2f}, {all_24hr_cells['flow_pc_score_all'].max():.2f}]")

# Calculate global percentile thresholds
low_pct_min = all_24hr_cells["flow_pc_score_all"].quantile(0.05)
low_pct_max = all_24hr_cells["flow_pc_score_all"].quantile(0.15)
high_pct_min = all_24hr_cells["flow_pc_score_all"].quantile(0.85)
high_pct_max = all_24hr_cells["flow_pc_score_all"].quantile(0.95)

print(f"\nGlobal percentile thresholds (based on both genotypes):")
print(f"  Low (5-15th): [{low_pct_min:.2f}, {low_pct_max:.2f}]")
print(f"  High (85-95th): [{high_pct_min:.2f}, {high_pct_max:.2f}]")

# Select cells for each genotype and percentile range
N_CELLS_PER_GROUP = 12  # 12 cells per genotype per percentile range

global_comparison_selections = {}

print("\nSelecting cells...")
print("-" * 80)

for genotype in ["WT", "V53I"]:
    # Low percentile
    low_cells = df.filter(
        (pl.col("genotype") == genotype) &
        (pl.col("treatment") == "24HR") &
        (pl.col("flow_pc_score_all") >= low_pct_min) &
        (pl.col("flow_pc_score_all") < low_pct_max)
    )
    
    if len(low_cells) >= N_CELLS_PER_GROUP:
        global_comparison_selections[f"{genotype}_Low"] = low_cells.sample(n=N_CELLS_PER_GROUP, seed=RANDOM_SEED)
        print(f"{genotype} Low: Selected {N_CELLS_PER_GROUP} cells (from {len(low_cells)} available)")
    else:
        global_comparison_selections[f"{genotype}_Low"] = low_cells
        print(f"{genotype} Low: Selected ALL {len(low_cells)} cells (< {N_CELLS_PER_GROUP} requested)")
    
    # High percentile
    high_cells = df.filter(
        (pl.col("genotype") == genotype) &
        (pl.col("treatment") == "24HR") &
        (pl.col("flow_pc_score_all") >= high_pct_min) &
        (pl.col("flow_pc_score_all") < high_pct_max)
    )
    
    if len(high_cells) >= N_CELLS_PER_GROUP:
        global_comparison_selections[f"{genotype}_High"] = high_cells.sample(n=N_CELLS_PER_GROUP, seed=RANDOM_SEED)
        print(f"{genotype} High: Selected {N_CELLS_PER_GROUP} cells (from {len(high_cells)} available)")
    else:
        global_comparison_selections[f"{genotype}_High"] = high_cells
        print(f"{genotype} High: Selected ALL {len(high_cells)} cells (< {N_CELLS_PER_GROUP} requested)")

print("\n" + "=" * 80)
print("Selection complete!")
print("=" * 80)

In [ ]:
def visualize_global_percentile_comparison(selections_dict, crop_size=1024, save_path=None):
    """
    Create comparison grid for WT vs V53I using global percentiles.
    
    Layout: 8 rows × 6 columns (12 cells per genotype per percentile range)
    - Rows 1-2: WT Low (5-15th percentile) - 12 cells
    - Rows 3-4: V53I Low (5-15th percentile) - 12 cells  
    - Rows 5-6: WT High (85-95th percentile) - 12 cells
    - Rows 7-8: V53I High (85-95th percentile) - 12 cells
    """
    # Define the order of groups
    groups = ["WT_Low", "V53I_Low", "WT_High", "V53I_High"]
    group_labels = [
        "WT - Low (5-15th)",
        "V53I - Low (5-15th)",
        "WT - High (85-95th)",
        "V53I - High (85-95th)"
    ]
    
    # Create figure: 8 rows × 6 columns
    # Each group gets 2 rows (12 cells: 2 rows × 6 columns)
    fig, axes = plt.subplots(8, 6, figsize=(18, 24))
    
    for group_idx, (group, label) in enumerate(zip(groups, group_labels)):
        cells = selections_dict[group]
        cells_list = list(cells.iter_rows(named=True))
        
        # Sort by flow score for consistent display
        cells_list.sort(key=lambda x: x["flow_pc_score_all"])
        
        # Each group occupies 2 rows
        start_row = group_idx * 2
        
        # Display 12 cells (2 rows × 6 columns)
        for cell_idx in range(12):
            row_offset = cell_idx // 6  # 0 or 1 (which of the 2 rows)
            col_idx = cell_idx % 6      # 0-5 (which column)
            row_idx = start_row + row_offset
            
            ax = axes[row_idx, col_idx]
            
            if cell_idx < len(cells_list):
                cell = cells_list[cell_idx]
                
                try:
                    # Find image file
                    img_filename = cell["FileName_CCM2"]
                    img_path = find_image_file(img_filename)
                    
                    if img_path is None:
                        ax.text(0.5, 0.5, "Image\nnot found", 
                               ha='center', va='center', fontsize=8)
                        ax.axis('off')
                        continue
                    
                    # Read all channels
                    channels = read_all_channels(img_path)
                    
                    # Create RGB composite
                    center_x = cell["Location_Center_X"]
                    center_y = cell["Location_Center_Y"]
                    rgb = create_rgb_composite(channels, center_x, center_y, crop_size)
                    
                    # Display image
                    ax.imshow(rgb)
                    
                    # Add metadata BELOW the image (outside the cell crop)
                    flow_score = cell["flow_pc_score_all"]
                    biorep = cell["biorep"]
                    
                    # Create metadata text
                    metadata_text = f"{biorep}\nScore: {flow_score:.1f}"
                    
                    # Display below the image
                    ax.text(0.5, -0.08, metadata_text,
                           transform=ax.transAxes,
                           ha='center', va='top',
                           fontsize=6,
                           bbox=dict(boxstyle='round,pad=0.3', 
                                   facecolor='lightgray', 
                                   alpha=0.8,
                                   edgecolor='none'))
                    
                    # Add row label on the left (first column, first row of each group only)
                    if col_idx == 0 and row_offset == 0:
                        genotype = cell["genotype"]
                        label_color = '#4169E1' if genotype == "WT" else '#FF8C00'
                        
                        ax.text(-0.15, 1.0, label,
                               transform=ax.transAxes,
                               rotation=90,
                               ha='right', va='center',
                               fontsize=10, fontweight='bold',
                               color=label_color)
                    
                except Exception as e:
                    print(f"Error processing cell: {e}")
                    ax.text(0.5, 0.5, "Error", ha='center', va='center', fontsize=8)
            
            ax.axis('off')
    
    # Overall title
    fig.suptitle('WT vs V53I Comparison - Global Percentiles (24HR Shear Stress)\n' +
                'Percentiles calculated from BOTH genotypes combined\n' +
                'All 5 Channels Merged (DAPI=Blue, WGA=Magenta, Phalloidin=Green, GM130=Red, CCM2=Orange)',
                fontsize=14, fontweight='bold', y=0.995)
    
    plt.tight_layout(rect=[0, 0, 1, 0.99])
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Saved to {save_path}")
    
    plt.show()

# Generate visualization
print("\nGenerating global percentile comparison visualization...")
visualize_global_percentile_comparison(
    global_comparison_selections,
    crop_size=CROP_SIZE,
    save_path=f"{OUTPUT_DIR}/WT_vs_V53I_24HR_global_percentiles.png"
)

## 7. Statistical Analysis: Density Plots

### Objective
Compare flow PC score distributions between WT and V53I across all three conditions with statistical testing.

In [ ]:
def plot_density_comparison(df, save_path=None, stats_csv_path=None):
    """
    Create density plots comparing WT vs V53I across conditions.
    Statistical results are saved separately to CSV instead of displayed on plot.
    
    Parameters:
    -----------
    df : polars.DataFrame
        Full dataset
    save_path : str or None
        Path to save the figure
    stats_csv_path : str or None
        Path to save statistical results CSV
    """
    conditions = ["STATIC", "2HR", "24HR"]
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # Store statistical results
    stats_results = []
    pvalues = []
    
    for idx, condition in enumerate(conditions):
        ax = axes[idx]
        
        # Get data for this condition
        wt_data = df.filter(
            (pl.col("genotype") == "WT") & 
            (pl.col("treatment") == condition)
        )["flow_pc_score_all"].to_numpy()
        
        v53i_data = df.filter(
            (pl.col("genotype") == "V53I") & 
            (pl.col("treatment") == condition)
        )["flow_pc_score_all"].to_numpy()
        
        # Plot densities
        ax.hist(wt_data, bins=50, alpha=0.6, color='#4169E1', 
               label=f'WT (n={len(wt_data)})', density=True, edgecolor='black', linewidth=0.5)
        ax.hist(v53i_data, bins=50, alpha=0.6, color='#FF8C00', 
               label=f'V53I (n={len(v53i_data)})', density=True, edgecolor='black', linewidth=0.5)
        
        # Add KDE overlay
        from scipy.stats import gaussian_kde
        
        if len(wt_data) > 1:
            kde_wt = gaussian_kde(wt_data)
            x_range = np.linspace(wt_data.min(), wt_data.max(), 200)
            ax.plot(x_range, kde_wt(x_range), color='#000080', linewidth=2, alpha=0.8)
        
        if len(v53i_data) > 1:
            kde_v53i = gaussian_kde(v53i_data)
            x_range = np.linspace(v53i_data.min(), v53i_data.max(), 200)
            ax.plot(x_range, kde_v53i(x_range), color='#CC7000', linewidth=2, alpha=0.8)
        
        # Statistical test
        t_stat, p_val = stats.ttest_ind(wt_data, v53i_data)
        pvalues.append(p_val)
        
        # Store statistics for CSV export
        stats_results.append({
            'Condition': condition,
            'WT_N': len(wt_data),
            'WT_Mean': wt_data.mean(),
            'WT_Std': wt_data.std(),
            'WT_Median': np.median(wt_data),
            'V53I_N': len(v53i_data),
            'V53I_Mean': v53i_data.mean(),
            'V53I_Std': v53i_data.std(),
            'V53I_Median': np.median(v53i_data),
            'Mean_Difference': v53i_data.mean() - wt_data.mean(),
            't_statistic': t_stat,
            'p_value': p_val
        })
        
        # Formatting
        ax.set_xlabel('Flow PC Score', fontsize=12, fontweight='bold')
        ax.set_ylabel('Density', fontsize=12, fontweight='bold')
        ax.set_title(condition, fontsize=14, fontweight='bold')
        ax.legend(loc='upper right', fontsize=10)
        ax.grid(True, alpha=0.3)
    
    # Multiple testing correction
    reject, pvals_corrected, _, _ = multipletests(pvalues, method='fdr_bh')
    
    # Add corrected p-values to results
    for i, result in enumerate(stats_results):
        result['p_value_FDR_corrected'] = pvals_corrected[i]
        result['Significant_FDR'] = reject[i]
    
    plt.suptitle('Flow PC Score Distribution: WT vs V53I Across Conditions',
                fontsize=16, fontweight='bold', y=1.02)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Saved plot to {save_path}")
    
    plt.show()
    
    # Save statistical results to CSV
    if stats_csv_path:
        stats_df = pd.DataFrame(stats_results)
        stats_df.to_csv(stats_csv_path, index=False, float_format='%.6f')
        print(f"Saved statistical results to {stats_csv_path}")
    
    # Print summary to console
    print("\n" + "="*80)
    print("STATISTICAL SUMMARY: WT vs V53I Comparison")
    print("="*80)
    for result in stats_results:
        print(f"\n{result['Condition']}:")
        print(f"  WT:   n={result['WT_N']:4d}, mean={result['WT_Mean']:7.2f} ± {result['WT_Std']:.2f}, median={result['WT_Median']:.2f}")
        print(f"  V53I: n={result['V53I_N']:4d}, mean={result['V53I_Mean']:7.2f} ± {result['V53I_Std']:.2f}, median={result['V53I_Median']:.2f}")
        print(f"  Difference: {result['Mean_Difference']:+.2f}")
        print(f"  t-statistic: {result['t_statistic']:.3f}")
        print(f"  p-value (raw): {result['p_value']:.4e}")
        print(f"  p-value (FDR-corrected): {result['p_value_FDR_corrected']:.4e}")
        sig_marker = "***" if result['p_value_FDR_corrected'] < 0.001 else "**" if result['p_value_FDR_corrected'] < 0.01 else "*" if result['p_value_FDR_corrected'] < 0.05 else "ns"
        print(f"  Significant: {result['Significant_FDR']} ({sig_marker})")
    print("="*80)
    
    return pd.DataFrame(stats_results)

# Generate density plots
stats_results_df = plot_density_comparison(
    df,
    save_path=f"{OUTPUT_DIR}/density_comparison_WT_vs_V53I.png",
    stats_csv_path=f"{OUTPUT_DIR}/stats_WT_vs_V53I_comparison.csv"
)

In [ ]:
def plot_wt_condition_comparison(df, save_path=None, stats_csv_path=None):
    """
    Create density plot comparing WT cells across the three conditions.
    Statistical results are saved separately to CSV instead of displayed on plot.
    
    Parameters:
    -----------
    df : polars.DataFrame
        Full dataset
    save_path : str or None
        Path to save the figure
    stats_csv_path : str or None
        Path to save statistical results CSV
    """
    conditions = ["STATIC", "2HR", "24HR"]
    colors = ['#2E86AB', '#A23B72', '#F18F01']  # Blue, Purple, Orange for STATIC, 2HR, 24HR
    
    fig, ax = plt.subplots(1, 1, figsize=(10, 6))
    
    # Collect data for each condition
    condition_data = {}
    descriptive_stats = []
    
    for condition in conditions:
        condition_data[condition] = df.filter(
            (pl.col("genotype") == "WT") & 
            (pl.col("treatment") == condition)
        )["flow_pc_score_all"].to_numpy()
        
        # Store descriptive statistics
        data = condition_data[condition]
        descriptive_stats.append({
            'Condition': condition,
            'N': len(data),
            'Mean': data.mean(),
            'Std': data.std(),
            'Median': np.median(data),
            'Q25': np.percentile(data, 25),
            'Q75': np.percentile(data, 75),
            'Min': data.min(),
            'Max': data.max()
        })
    
    # Plot histograms
    for idx, condition in enumerate(conditions):
        data = condition_data[condition]
        ax.hist(data, bins=50, alpha=0.4, color=colors[idx], 
               label=f'{condition} (n={len(data)})', density=True, 
               edgecolor='black', linewidth=0.5)
    
    # Add KDE overlays
    from scipy.stats import gaussian_kde
    
    for idx, condition in enumerate(conditions):
        data = condition_data[condition]
        if len(data) > 1:
            kde = gaussian_kde(data)
            x_range = np.linspace(data.min(), data.max(), 200)
            ax.plot(x_range, kde(x_range), color=colors[idx], 
                   linewidth=3, alpha=0.9)
    
    # Statistical tests
    # One-way ANOVA
    f_stat, anova_p = stats.f_oneway(
        condition_data["STATIC"],
        condition_data["2HR"],
        condition_data["24HR"]
    )
    
    # Pairwise t-tests
    pairwise_comparisons = [
        ("STATIC", "2HR"),
        ("STATIC", "24HR"),
        ("2HR", "24HR")
    ]
    
    pairwise_results = []
    pairwise_pvals = []
    
    for cond1, cond2 in pairwise_comparisons:
        t_stat, p_val = stats.ttest_ind(condition_data[cond1], condition_data[cond2])
        pairwise_pvals.append(p_val)
        
        mean1 = condition_data[cond1].mean()
        mean2 = condition_data[cond2].mean()
        
        pairwise_results.append({
            'Comparison': f"{cond1} vs {cond2}",
            f'{cond1}_Mean': mean1,
            f'{cond1}_Std': condition_data[cond1].std(),
            f'{cond2}_Mean': mean2,
            f'{cond2}_Std': condition_data[cond2].std(),
            'Mean_Difference': mean2 - mean1,
            't_statistic': t_stat,
            'p_value': p_val
        })
    
    # Multiple testing correction for pairwise comparisons
    reject, pvals_corrected, _, _ = multipletests(pairwise_pvals, method='bonferroni')
    
    # Add corrected p-values
    for i, result in enumerate(pairwise_results):
        result['p_value_Bonferroni_corrected'] = pvals_corrected[i]
        result['Significant_Bonferroni'] = reject[i]
    
    # Formatting
    ax.set_xlabel('Flow PC Score', fontsize=14, fontweight='bold')
    ax.set_ylabel('Density', fontsize=14, fontweight='bold')
    ax.set_title('WT Cells: Flow-Response Score Distribution Across Shear Stress Conditions',
                fontsize=15, fontweight='bold', pad=15)
    ax.legend(loc='upper left', fontsize=11, framealpha=0.9)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Saved plot to {save_path}")
    
    plt.show()
    
    # Save statistical results to CSV
    if stats_csv_path:
        # Create comprehensive stats DataFrame
        # Sheet 1: Descriptive statistics
        desc_df = pd.DataFrame(descriptive_stats)
        
        # Sheet 2: ANOVA result
        anova_df = pd.DataFrame([{
            'Test': 'One-way ANOVA',
            'F_statistic': f_stat,
            'p_value': anova_p
        }])
        
        # Sheet 3: Pairwise comparisons
        pairwise_df = pd.DataFrame(pairwise_results)
        
        # Save all to CSV (we'll save as separate CSVs since CSV doesn't support sheets)
        base_path = stats_csv_path.replace('.csv', '')
        desc_df.to_csv(f"{base_path}_descriptive.csv", index=False, float_format='%.6f')
        anova_df.to_csv(f"{base_path}_ANOVA.csv", index=False, float_format='%.6f')
        pairwise_df.to_csv(f"{base_path}_pairwise.csv", index=False, float_format='%.6f')
        
        print(f"Saved statistical results:")
        print(f"  - Descriptive stats: {base_path}_descriptive.csv")
        print(f"  - ANOVA results: {base_path}_ANOVA.csv")
        print(f"  - Pairwise comparisons: {base_path}_pairwise.csv")
    
    # Print summary to console
    print("\n" + "="*80)
    print("STATISTICAL SUMMARY: WT Condition Comparison")
    print("="*80)
    print(f"\nOne-way ANOVA: F = {f_stat:.3f}, p = {anova_p:.4e}")
    
    print("\nDescriptive Statistics:")
    print("-" * 80)
    for stat in descriptive_stats:
        print(f"{stat['Condition']:8s}: n={stat['N']:4d}, mean={stat['Mean']:7.2f} ± {stat['Std']:.2f}, "
              f"median={stat['Median']:.2f}, range=[{stat['Min']:.2f}, {stat['Max']:.2f}]")
    
    print("\nPairwise Comparisons (Bonferroni correction):")
    print("-" * 80)
    for result in pairwise_results:
        sig_marker = "***" if result['p_value_Bonferroni_corrected'] < 0.001 else "**" if result['p_value_Bonferroni_corrected'] < 0.01 else "*" if result['p_value_Bonferroni_corrected'] < 0.05 else "ns"
        print(f"{result['Comparison']} ({sig_marker}):")
        print(f"  Difference: {result['Mean_Difference']:+.2f}")
        print(f"  t-statistic: {result['t_statistic']:.3f}")
        print(f"  p-value (raw): {result['p_value']:.4e}")
        print(f"  p-value (Bonferroni): {result['p_value_Bonferroni_corrected']:.4e}")
        print(f"  Significant: {result['Significant_Bonferroni']}")
    print("="*80)
    
    return desc_df, anova_df, pairwise_df

# Generate WT-only condition comparison plot
desc_stats, anova_stats, pairwise_stats = plot_wt_condition_comparison(
    df,
    save_path=f"{OUTPUT_DIR}/density_comparison_WT_conditions.png",
    stats_csv_path=f"{OUTPUT_DIR}/stats_WT_conditions.csv"
)

## 8. Summary Statistics Table

In [ ]:
# Create summary statistics table
summary_stats = []

for genotype in ["WT", "V53I"]:
    for condition in ["STATIC", "2HR", "24HR"]:
        subset = df.filter(
            (pl.col("genotype") == genotype) & 
            (pl.col("treatment") == condition)
        )
        
        scores = subset["flow_pc_score_all"].to_numpy()
        
        summary_stats.append({
            "Genotype": genotype,
            "Condition": condition,
            "N_cells": len(scores),
            "Mean": np.mean(scores),
            "Std": np.std(scores),
            "Median": np.median(scores),
            "Min": np.min(scores),
            "Max": np.max(scores),
            "Q25": np.percentile(scores, 25),
            "Q75": np.percentile(scores, 75)
        })

summary_df = pd.DataFrame(summary_stats)
summary_df = summary_df.round(3)

# Save to CSV
summary_df.to_csv(f"{OUTPUT_DIR}/flow_score_summary_statistics.csv", index=False)
print(f"\nSummary statistics saved to {OUTPUT_DIR}/flow_score_summary_statistics.csv")

# Display
summary_df

## 9. Export Selected Cell Information

Save metadata for all selected cells for reproducibility.

In [ ]:
# Collect all selected cells from both visualizations
all_selected = []

# From WT flow response visualization (new structure)
for pct_label, conditions_dict in wt_selections.items():
    for condition, cells in conditions_dict.items():
        for cell in cells.iter_rows(named=True):
            all_selected.append({
                "visualization_set": "WT_flow_response",
                "genotype": "WT",
                "condition": condition,
                "percentile_range": pct_label,
                "cell_id": cell["cell_id"],
                "flow_pc_score": cell["flow_pc_score_all"],
                "biorep": cell["biorep"],
                "ImageNumber": cell["ImageNumber"],
                "ObjectNumber": cell["ObjectNumber"],
                "Location_Center_X": cell["Location_Center_X"],
                "Location_Center_Y": cell["Location_Center_Y"],
                "FileName_CCM2": cell["FileName_CCM2"]
            })

# From WT vs V53I comparison
for group_name, cells in comparison_selections.items():
    for cell in cells.iter_rows(named=True):
        all_selected.append({
            "visualization_set": "WT_vs_V53I_24HR",
            "genotype": cell["genotype"],
            "condition": "24HR",
            "percentile_group": group_name,
            "cell_id": cell["cell_id"],
            "flow_pc_score": cell["flow_pc_score_all"],
            "biorep": cell["biorep"],
            "ImageNumber": cell["ImageNumber"],
            "ObjectNumber": cell["ObjectNumber"],
            "Location_Center_X": cell["Location_Center_X"],
            "Location_Center_Y": cell["Location_Center_Y"],
            "FileName_CCM2": cell["FileName_CCM2"]
        })

# Convert to DataFrame and save
selected_cells_df = pd.DataFrame(all_selected)
selected_cells_df.to_csv(f"{OUTPUT_DIR}/selected_cells_metadata.csv", index=False)

print(f"\nExported metadata for {len(selected_cells_df)} selected cells")
print(f"Saved to {OUTPUT_DIR}/selected_cells_metadata.csv")

selected_cells_df.head(10)

## 10. Session Info

In [ ]:
import sys
from datetime import datetime

print("Session Information")
print("=" * 60)
print(f"Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Python version: {sys.version}")
print(f"\nPackage versions:")
print(f"  polars: {pl.__version__}")
print(f"  numpy: {np.__version__}")
print(f"  matplotlib: {mpl.__version__}")
print(f"  seaborn: {sns.__version__}")
print(f"\nOutput directory: {OUTPUT_DIR}")
print(f"Random seed: {RANDOM_SEED}")
print(f"Crop size: {CROP_SIZE} pixels")